# Study sites and thickness mapping

This Python notebook generates high-resolution, interpolated 2D ice thickness maps for the Alphubel, Chessjen, and Hohsaas glaciers. It utilizes preprocessed GPR (Ground Penetrating Radar) bedrock pick data, overlays the results on orthophotos automatically downloaded from Swisstopo, and visualizes borehole locations on each map. The workflow includes data loading, spatial interpolation, and advanced cartographic visualization, providing a comprehensive overview of glacier ice thickness in relation to surface features and borehole sites.

The data has already been preprocessed by Dr. Christophe Ogier & Dr. Ilaria Santin. Several steps have been conducted:

1. Filtering
2. Bed picking
3. etc.


In [ ]:
%matplotlib inline
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 100  # keeps inline figures small for version control

import sys

# Add project root to Python path
project_root = '/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Cell 1: Imports and inputs
import os
import geopandas as gpd
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import gpr_processing as gpr
import gpr_plotting as gprp
import rasterio
import cmcrameri.cm as cmc
from matplotlib.colors import BoundaryNorm
from processing.thermistor_processing import *
from processing.plot_composer import *
from processing.geodata_processing import *
from rasterio.plot import show as rioshow


pixel_size = 20.0    # meters
interp_method = 'linear'  # 'linear' | 'cubic' | 'nearest'

### Set up paths

In [ ]:
root_dir = "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/gpr/"
project_dir = "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers"

# List GPR bedrock picks files for each glacier/profile
gpr_picks_dirs = [
    root_dir + "/20250515_Alphubel/bed picks_south/20060101_GPR_picks_south.txt", # Alphubel south
    root_dir + "/20240809_Chessjen/picks/20240809_chessjen_picks_updated.csv", # Chessjen
    root_dir + "/20240808_Hohsaas/picks/20240808_hohsaas_picks.csv", # Hohsaas (if available)
    root_dir + "/20240806_SexRouge/picks/Horizon_interpretation_SR_formatted.csv", # Sex Rouge
    root_dir + "/20240807_Tortin/picks/Horizon_interpretation_GT_formatted.csv", # Tortin
    root_dir + "/20240828_Corvatsch/picks/Horizon_interpretation_CV_formatted.csv" # Corvatsch
]

# Optional local orthophoto (GeoTIFF). Leave None to use online basemap instead.
ortho_path = None  # e.g., os.path.join(root_dir, "orthophoto.tif")

In [ ]:
# load GPR points from TXT files (force LV95 / EPSG:2056) --> represent gpr lines as points
gpr_lines_AH = gpr.load_points_from_txt(gpr_picks_dirs[0], epsg=2056, drop_duplicates=True, aggregate_duplicates='mean')

# load GPR points from CSV files (force LV95 / EPSG:2056) --> represent gpr lines as points
gpr_lines_CJ = gpr.load_points_from_csv(gpr_picks_dirs[1], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean')
gpr_lines_HS = gpr.load_points_from_csv(gpr_picks_dirs[2], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean')
gpr_lines_SR = gpr.load_points_from_csv(gpr_picks_dirs[3], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean')
gpr_lines_GT = gpr.load_points_from_csv(gpr_picks_dirs[4], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean')
gpr_lines_CV = gpr.load_points_from_csv(gpr_picks_dirs[5], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean')

# load GPR points from shp files (force LV95 / EPSG:2056) --> represent gpr lines as points
# gpr_lines_SR = gpr.load_points_from_shps_dir(gpr_picks_dirs[3], epsg=2056, drop_duplicates=True, aggregate_duplicates='mean')
# gpr_lines_GT = gpr.load_points_from_shps_dir(gpr_picks_dirs[4], epsg=2056, drop_duplicates=True, aggregate_duplicates='mean')
# gpr_lines_CV = gpr.load_points_from_shps_dir(gpr_picks_dirs[5], epsg=2056, drop_duplicates=True, aggregate_duplicates='mean')

# Create coverage polygon from GPR lines
coverage_AH = gpr.make_coverage_polygon(gpr_lines_AH, method='convex', buffer_m=0.0)  # avoids optimizealpha
print(gpr_lines_AH.shape, coverage_AH.crs, coverage_AH.geometry.iloc[0].geom_type)

coverage_CJ = gpr.make_coverage_polygon(gpr_lines_CJ, method='convex', buffer_m=0.0)  # avoids optimizealpha
print(gpr_lines_CJ.shape, coverage_CJ.crs, coverage_CJ.geometry.iloc[0].geom_type)

coverage_HS = gpr.make_coverage_polygon(gpr_lines_HS, method='convex', buffer_m=0.0)  # avoids optimizealpha
print(gpr_lines_HS.shape, coverage_HS.crs, coverage_HS.geometry.iloc[0].geom_type)

coverage_SR = gpr.make_coverage_polygon(gpr_lines_SR, method='convex', buffer_m=0.0)  # avoids optimizealpha
print(gpr_lines_SR.shape, coverage_SR.crs, coverage_SR.geometry.iloc[0].geom_type)

coverage_GT = gpr.make_coverage_polygon(gpr_lines_GT, method='convex', buffer_m=0.0)  # avoids optimizealpha
print(gpr_lines_GT.shape, coverage_GT.crs, coverage_GT.geometry.iloc[0].geom_type)

coverage_CV = gpr.make_coverage_polygon(gpr_lines_CV, method='convex', buffer_m=0.0)  # avoids optimizealpha
print(gpr_lines_CV.shape, coverage_CV.crs, coverage_CV.geometry.iloc[0].geom_type)

### Download Orhophotos from Swisstopo

In [ ]:
# set download dir
ortho_dir = project_dir + "/products/figures/gpr_figures/ice_thickness_maps/"

# Buffer around coverage and per-glacier pixel sizes
ortho_buffer_m = 100
px = 0.25

# Square bboxes (centered on coverage) snapped to pixel grid
bbox_AH = gpr.square_bbox_from_gdf(coverage_AH, buffer_m=ortho_buffer_m, pixel_size=px)
bbox_CJ = gpr.square_bbox_from_gdf(coverage_CJ, buffer_m=ortho_buffer_m, pixel_size=px)
bbox_HS = gpr.square_bbox_from_gdf(coverage_HS, buffer_m=ortho_buffer_m, pixel_size=px)

bbox_SR = gpr.square_bbox_from_gdf(coverage_SR, buffer_m=ortho_buffer_m, pixel_size=px)
bbox_GT = gpr.square_bbox_from_gdf(coverage_GT, buffer_m=ortho_buffer_m, pixel_size=px)
bbox_CV = gpr.square_bbox_from_gdf(coverage_CV, buffer_m=ortho_buffer_m, pixel_size=px)

# Download orthophotos
out_ortho_AH = os.path.join(ortho_dir, "alphubel_orthophoto.tif")
_, ortho_transform_AH, ortho_crs_AH = gpr.download_swisstopo_orthophoto(
    bbox_AH, out_ortho_AH, crs_epsg=2056, pixel_size=px,
    layer="ch.swisstopo.swissimage", fmt="image/jpeg"
)
print("Alphubel orthophoto saved:", out_ortho_AH)

out_ortho_CJ = os.path.join(ortho_dir, "chessjen_orthophoto.tif")
_, ortho_transform_CJ, ortho_crs_CJ = gpr.download_swisstopo_orthophoto(
    bbox_CJ, out_ortho_CJ, crs_epsg=2056, pixel_size=px,
    layer="ch.swisstopo.swissimage", fmt="image/jpeg"
)
print("Chessjen orthophoto saved:", out_ortho_CJ)

out_ortho_HS = os.path.join(ortho_dir, "hohsaas_orthophoto.tif")
_, ortho_transform_HS, ortho_crs_HS = gpr.download_swisstopo_orthophoto(
    bbox_HS, out_ortho_HS, crs_epsg=2056, pixel_size=px,
    layer="ch.swisstopo.swissimage", fmt="image/jpeg"
)
print("Hohsaas orthophoto saved:", out_ortho_HS)

out_ortho_SR = os.path.join(ortho_dir, "sexrouge_orthophoto.tif")
_, ortho_transform_SR, ortho_crs_SR = gpr.download_swisstopo_orthophoto(
    bbox_SR, out_ortho_SR, crs_epsg=2056, pixel_size=px,
    layer="ch.swisstopo.swissimage", fmt="image/jpeg"
)
print("Sex Rouge orthophoto saved:", out_ortho_SR)

out_ortho_GT = os.path.join(ortho_dir, "tortin_orthophoto.tif")
_, ortho_transform_GT, ortho_crs_GT = gpr.download_swisstopo_orthophoto(
    bbox_GT, out_ortho_GT, crs_epsg=2056, pixel_size=px,
    layer="ch.swisstopo.swissimage", fmt="image/jpeg"
)
print("Tortin orthophoto saved:", out_ortho_GT)

out_ortho_CV = os.path.join(ortho_dir, "corvatsch_orthophoto.tif")
_, ortho_transform_CV, ortho_crs_CV = gpr.download_swisstopo_orthophoto(
    bbox_CV, out_ortho_CV, crs_epsg=2056, pixel_size=px,
    layer="ch.swisstopo.swissimage", fmt="image/jpeg"
)
print("Corvatsch orthophoto saved:", out_ortho_CV)

### Interpolate GPR ice thickness to grid

In [ ]:
# Interpolation settings
pixel_size = 3.0    # meters (was 20.0)
rbf_function = 'linear'  # or 'multiquadric', 'thin_plate', etc.

# Interpolate GPR thickness to grid for ALPHUBEL
grid_AH, grid_transform_AH, grid_crs_AH = gpr.interpolate_thickness_to_grid(
    gpr_lines_AH, value_col='thickness',
    pixel_size=pixel_size, rbf_function=rbf_function,
    polygon_mask=coverage_AH, padding=0.0
)
print("Alphubel grid shape:", grid_AH.shape)

# Interpolate GPR thickness to grid for CHESSJEN
grid_CJ, grid_transform_CJ, grid_crs_CJ = gpr.interpolate_thickness_to_grid(
    gpr_lines_CJ, value_col='thickness',
    pixel_size=pixel_size, rbf_function=rbf_function,
    polygon_mask=coverage_CJ, padding=0.0
)
print("Chessjen grid shape:", grid_CJ.shape)

# Interpolate GPR thickness to grid for HOHSAAS
grid_HS, grid_transform_HS, grid_crs_HS = gpr.interpolate_thickness_to_grid(
    gpr_lines_HS, value_col='thickness',
    pixel_size=pixel_size, rbf_function=rbf_function,
    polygon_mask=coverage_HS, padding=0.0
)
print("Hohsaas grid shape:", grid_HS.shape)

# Interpolate GPR thickness to grid for SEX ROUGE
grid_SR, grid_transform_SR, grid_crs_SR = gpr.interpolate_thickness_to_grid(
    gpr_lines_SR, value_col='thickness',
    pixel_size=pixel_size, rbf_function=rbf_function,
    polygon_mask=coverage_SR, padding=0.0
)
print("Sex Rouge grid shape:", grid_SR.shape)

# Interpolate GPR thickness to grid for TORTIN
grid_GT, grid_transform_GT, grid_crs_GT = gpr.interpolate_thickness_to_grid(
    gpr_lines_GT, value_col='thickness',
    pixel_size=pixel_size, rbf_function=rbf_function,   
    polygon_mask=coverage_GT, padding=0.0
)
print("Tortin grid shape:", grid_GT.shape)

# Interpolate GPR thickness to grid for CORVATSCH
grid_CV, grid_transform_CV, grid_crs_CV = gpr.interpolate_thickness_to_grid(
    gpr_lines_CV, value_col='thickness',
    pixel_size=pixel_size, rbf_function=rbf_function,   
    polygon_mask=coverage_CV, padding=0.0
)
print("Corvatsch grid shape:", grid_CV.shape)

### Load other data for map

In [ ]:
# Load latest borehole positions
bh_csv = root_dir + "../icetemperature_data/thermistor_borehole_settings/thermistor_coordinates.csv"

# ALPHUBEL boreholes
bh_AH = ["AH1G", "AH2G", "AH3G", "AH1TT", "AH2TT", "AH3TT"]
boreholes_AH, bh_missing_AH = gpr.load_borehole_positions(bh_csv, keep_names=bh_AH)

# CHESSJEN boreholes
bh_CJ = ["CJ1G", "CJ2G", "CJ1TT", "CJ2TT", "CJ3TT", "CJ4TT"]  # Add other Chessjen boreholes if they exist
boreholes_CJ, bh_missing_CJ = gpr.load_borehole_positions(bh_csv, keep_names=bh_CJ)

# HOHSAAS boreholes
bh_HS = ["HS1G", "HS2G", "HS3G", "HS1TT", "HS2TT", "HS3TT"]  # Add other Hohsaas boreholes if they exist
boreholes_HS, bh_missing_HS = gpr.load_borehole_positions(bh_csv, keep_names=bh_HS)

# SEX ROUGE boreholes
bh_SR = ["SR1TT", "SR2TT"]  # Add other Sex Rouge boreholes if they exist
boreholes_SR, bh_missing_SR = gpr.load_borehole_positions(bh_csv, keep_names=bh_SR)

# TORTIN boreholes
bh_GT = ["GT1TT", "GT2TT"]  # Add other Tortin boreholes if they exist
boreholes_GT, bh_missing_GT = gpr.load_borehole_positions(bh_csv, keep_names=bh_GT)

# CORVATSCH boreholes
bh_CV = ["CV1TT", "CV2TT"]  # Add other Corvatsch boreholes if they exist
boreholes_CV, bh_missing_CV = gpr.load_borehole_positions(bh_csv, keep_names=bh_CV)

print(f"Alphubel boreholes loaded: {len(boreholes_AH)}")
print(f"Chessjen boreholes loaded: {len(boreholes_CJ)}")
print(f"Hohsaas boreholes loaded: {len(boreholes_HS)}")
print(f"Sex Rouge boreholes loaded: {len(boreholes_SR)}")
print(f"Tortin boreholes loaded: {len(boreholes_GT)}")
print(f"Corvatsch boreholes loaded: {len(boreholes_CV)}")

# Download SwissALTI3D DEM to generate elevation contours
from glob import glob

dems_dir = "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/products/figures/gpr_figures/ice_thickness_maps/dems"

# ALPHUBEL DEM tiles
dem_tiles_AH = sorted(glob(os.path.join(
    dems_dir, "alphubel", "*.tif")
))

# CHESSJEN DEM tiles
dem_tiles_CJ = sorted(glob(os.path.join(
    dems_dir, "chessjen", "*.tif")
))

# HOHSAAS DEM tiles
dem_tiles_HS = sorted(glob(os.path.join(
    dems_dir, "hohsaas", "*.tif")
))

# SEX ROUGE DEM tiles
dem_tiles_SR = sorted(glob(os.path.join(
    dems_dir, "sex_rouge", "*.tif")
))

# TORTIN DEM tiles
dem_tiles_GT = sorted(glob(os.path.join(
    dems_dir, "tortin", "*.tif")
))

# CORVATSCH DEM tiles
dem_tiles_CV = sorted(glob(os.path.join(
    dems_dir, "corvatsch", "*.tif")
))

In [ ]:
dem_tiles_AH

In [ ]:
boreholes_CJ

### Add column with bolean that shows whether borehole reached bedrock depth

In [ ]:
# Define which boreholes reached bedrock for each glacier
bedrock_reached = {
    'AH': ['AH1G', 'AH2G', 'AH3G'],  # Update with actual boreholes that reached bedrock
    'CJ': ['CJ1G', 'CJ2G'],
    'HS': ['HS1G', 'HS2G'],  # Update with actual boreholes
    'SR': [],  # Update as needed
    'GT': [],  # Update as needed
    'CV': ['CV1TT', 'CV2TT']   # Update as needed
}

# Add 'reached_bedrock' column to each borehole dataframe
boreholes_AH['reached_bedrock'] = boreholes_AH['name'].isin(bedrock_reached['AH'])
boreholes_CJ['reached_bedrock'] = boreholes_CJ['name'].isin(bedrock_reached['CJ'])
boreholes_HS['reached_bedrock'] = boreholes_HS['name'].isin(bedrock_reached['HS'])
boreholes_SR['reached_bedrock'] = boreholes_SR['name'].isin(bedrock_reached['SR'])
boreholes_GT['reached_bedrock'] = boreholes_GT['name'].isin(bedrock_reached['GT'])
boreholes_CV['reached_bedrock'] = boreholes_CV['name'].isin(bedrock_reached['CV'])

# Display Chessjen boreholes as example
print("\nChessjen boreholes with bedrock status:")
print(boreholes_CJ[['name', 'borehole depth (m)', 'reached_bedrock']])

In [ ]:
boreholes_AH

### Set label abbreviations

In [ ]:
abbr = {"Alphubel": "AH", "Chessjen": "CJ", "Hohsaas": "HS", "Sex Rouge": "SR", "Tortin": "GT", "Corvatsch": "CT"}
rect_colors = {"AH": "#1f77b4", "CJ": "#2ca02c", "HS": "#ff7f0e", "SR": "#d62728", "GT": "#9467bd", "CT": "#8c564b"}

### Download overview map for glacier location

In [ ]:
# Choose two LV95 coordinates (x, y)
# LV95 coordinates for the custom square (as floats, no thousands separator)
p1 = (2631593.19, 1100141.96)
p2 = (2644135.67, 1110169.50)

out_tif = os.path.join(ortho_dir, "custom_square_orthophoto.tif")
_, tfm, crs = gpr.download_swisstopo_orthophoto_from_points(
    p1, p2, out_tif,
    crs_epsg=2056, pixel_size=10.0,  # adjust resolution
    buffer_m=0.0                    # optional extra margin
)
print("Saved:", out_tif)

output_path = "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/products/figures/gpr_figures/ice_thickness_maps/"

ABBR_FONTSIZE = 16     # fontsize of the 'AH / CJ / HS' abbreviation tags
ANNO_FONTSIZE = 12     # fontsize of the borehole annotation labels

## Merge overview map with single plots & legend

### Add SGI 2023 outlines to the mosaic map

In [ ]:
from processing.geodata_processing import *

# set paths to .xyzn files
outline_dir = "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/data/sgi_2022/xyzn_lv95/"

xyzn_path_AH = outline_dir + "SGI_2023_B55-15_lv95.xyzn" # Alphubel
xyzn_path_CJ = outline_dir + "SGI_2023_B53-14_lv95.xyzn" # Chessjen
xyzn_path_HS = outline_dir + "SGI_2023_B51-13_lv95.xyzn" # Hohsaas
xyzn_path_SR = outline_dir + "SGI_2023_B16-01_lv95.xyzn" # Sex Rouge
xyzn_path_TT = outline_dir + "SGI_2023_B75-12_lv95.xyzn" # Tortin
xyzn_path_CV = outline_dir + "SGI_2022_E23-18_lv95.xyzn" # Corvatsch

# Load glacier outlines as GeoDataFrames
outline_AH = read_xyzn_to_gdf(xyzn_path_AH)
outline_CJ = read_xyzn_to_gdf(xyzn_path_CJ)
outline_HS = read_xyzn_to_gdf(xyzn_path_HS)
outline_SR = read_xyzn_to_gdf(xyzn_path_SR)
outline_TT = read_xyzn_to_gdf(xyzn_path_TT)
outline_CV = read_xyzn_to_gdf(xyzn_path_CV)

# set output figure directory
out_fig = os.path.join(output_path, "field_sites_overview_map.png")

### Version 2 all six glaciers HS, AH, CH, SR, GT, CV

In [ ]:
# New standalone notebook cell: 3-row mosaic (overview + legend, row of b/c/d, row of e/f/g)
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

overview_path = "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/products/figures/maps/field_sites_extended_overview.png"

abbr = {"Alphubel": "AH", "Chessjen": "CJ", "Hohsaas": "HS", "Sex Rouge": "SR", "Tortin": "GT", "Corvatsch": "CV"}
rect_colors = {"AH": "#1f77b4", "CJ": "#2ca02c", "HS": "#ff7f0e", "SR": "#d62728", "GT": "#9467bd", "CV": "#8c564b"}

ANNO_FONTSIZE = 12
ABBR_FONTSIZE = 12

fig = plt.figure(figsize=(13, 12), dpi=300)
gs = fig.add_gridspec(
    3, 3,
    height_ratios=[0.6, 1.0, 1.0],
    hspace=0.0
)

# extent for overview (same as previously used)
extent = (np.float64(2481120.0), np.float64(2838080.0), np.float64(1071040.0), np.float64(1300160.0))

# Panel a: Overview spanning cols 0:2
ax_a = fig.add_subplot(gs[0, 0:2])
_, _, _ = plot_cropped_map_with_grid(
    overview_path,
    extent,
    crop_top_px=900,
    grid_interval_km=50,
    ax=ax_a,
    fontsize=ANNO_FONTSIZE,
    xlabel=False,
    ylabel=False,
)
ax_a.text(0.01, 0.96, "(a)", transform=ax_a.transAxes, ha='left', va='top',
          fontsize=ANNO_FONTSIZE, fontweight='bold', color='black',
          bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=1.0), zorder=21)

# Legend in top-right slot (consistent vertical spacing)
ax_legend = fig.add_subplot(gs[0, 2])
ax_legend.set_axis_off()
ax_legend.set_xlim(0, 1); ax_legend.set_ylim(0, 1)

sym_x = 0.1; text_x = 0.2; y0 = 0.75; dy = 0.12

# Title
ax_legend.text(0.2, 0.9, "Legend", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, fontweight='bold', va='top', ha='center')

# Positions for legend entries (ensures equal vertical spacing)
y_positions = [y0 - i * dy for i in range(6)]

# Glacier area (blue square)
ax_legend.add_patch(Rectangle((sym_x - 0.015, y_positions[0] - 0.02), 0.03, 0.03, transform=ax_legend.transAxes,
                             facecolor='#8fb5e0', edgecolor='k', linewidth=0.8, zorder=2))
ax_legend.text(text_x, y_positions[0], "Glacierized area (SGI 2023)", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, va='center', ha='left')

# Fieldwork glacier (diamond)
ax_legend.scatter([sym_x], [y_positions[1]], s=90, marker='D', color='red',
                  edgecolors='black', linewidths=1.0, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[1], "Study site", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, va='center', ha='left')


# Geoprecision borehole (filled red circle)
ax_legend.scatter([sym_x], [y_positions[2]], s=90, marker='o', color='red',
                  edgecolors='black', linewidths=1.5, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[2], "Geoprecision borehole", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# TinyTag borehole (white circle)
ax_legend.scatter([sym_x], [y_positions[3]], s=90, marker='o', color='white',
                  edgecolors='black', linewidths=1.5, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[3], "TinyTag borehole", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# GPR profile (line)
ax_legend.plot([sym_x - 0.06, sym_x + 0.06], [y_positions[4], y_positions[4]], color='k', lw=2,
               solid_capstyle='round', transform=ax_legend.transAxes)
ax_legend.text(text_x, y_positions[4], "GPR profile", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# Glacier outline legend element (SGI 2023)
ax_legend.plot([sym_x - 0.06, sym_x + 0.06], [y_positions[5], y_positions[5]],
               color='k', lw=2, linestyle=':', transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[5], "Glacier outline (SGI 2023)", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# Add north arrow
add_north_arrow_compass(ax_legend, xy=(0.92, 0.85), size=0.1, colors=('white', '0.35'), edge='0.35', text_color='0.35', lw=1.0)

# Middle row panels b, c, d (Hohsaas, Alphubel, Chessjen)
axs_mid = [fig.add_subplot(gs[1, i]) for i in range(3)]
draw_glacier_map(axs_mid[0], out_ortho_HS, bbox_HS, gpr_lines_HS, dem_tiles_HS, boreholes_HS, "Hohsaas", xlabel=False, ylabel=False, panel='b', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_HS)
draw_glacier_map(axs_mid[1], out_ortho_AH, bbox_AH, gpr_lines_AH, dem_tiles_AH, boreholes_AH, "Alphubel", xlabel=False, ylabel=False, panel='c', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_AH)
draw_glacier_map(axs_mid[2], out_ortho_CJ, bbox_CJ, gpr_lines_CJ, dem_tiles_CJ, boreholes_CJ, "Chessjen", xlabel=False, ylabel=False, panel='d', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_CJ)

# Bottom row panels e, f, g (Sex Rouge, Tortin, Corvatsch)
axs_bot = [fig.add_subplot(gs[2, i]) for i in range(3)]
draw_glacier_map(axs_bot[0], out_ortho_SR, bbox_SR, gpr_lines_SR, dem_tiles_SR, boreholes_SR, "Sex Rouge", xlabel=False, ylabel=False, panel='e', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_SR)
draw_glacier_map(axs_bot[1], out_ortho_GT, bbox_GT, gpr_lines_GT, dem_tiles_GT, boreholes_GT, "Tortin", xlabel=False, ylabel=False, panel='f', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_TT)
draw_glacier_map(axs_bot[2], out_ortho_CV, bbox_CV, gpr_lines_CV, dem_tiles_CV, boreholes_CV, "Corvatsch", xlabel=False, ylabel=False, panel='g', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_CV)

plt.tight_layout()

# Add shared labels (choose fontsize to match other annotations)
fig.text(0.55, 0.1, "Easting (LV95) [m]", ha='center', va='bottom', fontsize=12)
fig.text(0.1, 0.55, "Northing (LV95) [m]", ha='center', va='center', rotation='vertical', fontsize=12)

# Add extra whitespace to left and bottom so labels remain visible after tight_layout
fig.subplots_adjust(left=0.13, bottom=0.13)

out_pdf = str(Path(out_fig).with_suffix('.pdf'))
plt.savefig(out_pdf, dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

### Version 3 all six glaciers HS, AH, CH, SR, GT, CV + glacier outlines

In [ ]:
# New standalone notebook cell: 3-row mosaic (overview + legend, row of b/c/d, row of e/f/g)
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

abbr = {"Alphubel": "AH", "Chessjen": "CJ", "Hohsaas": "HS", "Sex Rouge": "SR", "Tortin": "GT", "Corvatsch": "CV"}
rect_colors = {"AH": "#1f77b4", "CJ": "#2ca02c", "HS": "#ff7f0e", "SR": "#d62728", "GT": "#9467bd", "CV": "#8c564b"}

ANNO_FONTSIZE = 12
ABBR_FONTSIZE = 12

fig = plt.figure(figsize=(13, 12), dpi=300)
gs = fig.add_gridspec(
    3, 3,
    height_ratios=[0.6, 1.0, 1.0],
    hspace=0.0
)

# extent for overview (same as previously used)
extent = (np.float64(2481120.0), np.float64(2838080.0), np.float64(1071040.0), np.float64(1300160.0))

# Panel a: Overview spanning cols 0:2
ax_a = fig.add_subplot(gs[0, 0:2])
_, _, _ = plot_cropped_map_with_grid(
    overview_path,
    extent,
    crop_top_px=900,
    grid_interval_km=50,
    ax=ax_a,
    fontsize=ANNO_FONTSIZE,
    xlabel=False,
    ylabel=False,
)
ax_a.text(0.01, 0.96, "(a)", transform=ax_a.transAxes, ha='left', va='top',
          fontsize=ANNO_FONTSIZE, fontweight='bold', color='black',
          bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=1.0), zorder=21)

# Legend in top-right slot (consistent vertical spacing)
ax_legend = fig.add_subplot(gs[0, 2])
ax_legend.set_axis_off()
ax_legend.set_xlim(0, 1); ax_legend.set_ylim(0, 1)

sym_x = 0.1; text_x = 0.2; y0 = 0.75; dy = 0.12

# Title
ax_legend.text(0.2, 0.9, "Legend", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, fontweight='bold', va='top', ha='center')

# Positions for legend entries (ensures equal vertical spacing)
y_positions = [y0 - i * dy for i in range(6)]

# Glacier area (blue square)
ax_legend.add_patch(Rectangle((sym_x - 0.015, y_positions[0] - 0.02), 0.03, 0.03, transform=ax_legend.transAxes,
                             facecolor='#8fb5e0', edgecolor='k', linewidth=0.8, zorder=2))
ax_legend.text(text_x, y_positions[0], "Glacierized area (SGI 2023)", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, va='center', ha='left')

# Fieldwork glacier (diamond)
ax_legend.scatter([sym_x], [y_positions[1]], s=90, marker='D', color='red',
                  edgecolors='black', linewidths=1.0, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[1], "Study site", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, va='center', ha='left')

# Geoprecision borehole (filled red circle)
ax_legend.scatter([sym_x], [y_positions[2]], s=90, marker='o', color='red',
                  edgecolors='black', linewidths=1.5, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[2], "Geoprecision borehole", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# TinyTag borehole (white circle)
ax_legend.scatter([sym_x], [y_positions[3]], s=90, marker='o', color='white',
                  edgecolors='black', linewidths=1.5, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[3], "TinyTag borehole", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# GPR profile (line)
ax_legend.plot([sym_x - 0.06, sym_x + 0.06], [y_positions[4], y_positions[4]], color='k', lw=2,
               solid_capstyle='round', transform=ax_legend.transAxes)
ax_legend.text(text_x, y_positions[4], "GPR profile", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# Glacier outline legend element (SGI 2023)
ax_legend.plot([sym_x - 0.06, sym_x + 0.06], [y_positions[5], y_positions[5]],
               color='k', lw=2, linestyle=':', transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[5], "Glacier outline (SGI 2023)", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# Add north arrow
add_north_arrow_compass(ax_legend, xy=(0.92, 0.85), size=0.1, colors=('white', '0.35'), edge='0.35', text_color='0.35', lw=1.0)

# Middle row panels b, c, d (Hohsaas, Alphubel, Chessjen)
axs_mid = [fig.add_subplot(gs[1, i]) for i in range(3)]
draw_glacier_map(axs_mid[0], out_ortho_HS, bbox_HS, gpr_lines_HS, dem_tiles_HS, boreholes_HS, "Hohsaas", xlabel=False, ylabel=False, panel='b', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_HS, background='hillshade')
draw_glacier_map(axs_mid[1], out_ortho_AH, bbox_AH, gpr_lines_AH, dem_tiles_AH, boreholes_AH, "Alphubel", xlabel=False, ylabel=False, panel='c', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_AH, background='hillshade')
draw_glacier_map(axs_mid[2], out_ortho_CJ, bbox_CJ, gpr_lines_CJ, dem_tiles_CJ, boreholes_CJ, "Chessjen", xlabel=False, ylabel=False, panel='d', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_CJ, background='hillshade')

# Bottom row panels e, f, g (Sex Rouge, Tortin, Corvatsch)
axs_bot = [fig.add_subplot(gs[2, i]) for i in range(3)]
draw_glacier_map(axs_bot[0], out_ortho_SR, bbox_SR, gpr_lines_SR, dem_tiles_SR, boreholes_SR, "Sex Rouge", xlabel=False, ylabel=False, panel='e', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_SR, background='hillshade')
draw_glacier_map(axs_bot[1], out_ortho_GT, bbox_GT, gpr_lines_GT, dem_tiles_GT, boreholes_GT, "Tortin", xlabel=False, ylabel=False, panel='f', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_TT, background='hillshade')
draw_glacier_map(axs_bot[2], out_ortho_CV, bbox_CV, gpr_lines_CV, dem_tiles_CV, boreholes_CV, "Corvatsch", xlabel=False, ylabel=False, panel='g', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_CV, background='hillshade')

plt.tight_layout()


# Add shared labels (choose fontsize to match other annotations)
fig.text(0.55, 0.1, "Easting (LV95) [m]", ha='center', va='bottom', fontsize=12)
fig.text(0.1, 0.55, "Northing (LV95) [m]", ha='center', va='center', rotation='vertical', fontsize=12)

# Add extra whitespace to left and bottom so labels remain visible after tight_layout
fig.subplots_adjust(left=0.13, bottom=0.13)

# After all draw_glacier_map calls and before plt.show()
for ax, bbox in zip(axs_mid + axs_bot, [bbox_HS, bbox_AH, bbox_CJ, bbox_SR, bbox_GT, bbox_CV]):
    add_panel_outline(ax, bbox, color='black', linewidth=1.4)

out_pdf = str(Path(out_fig).with_suffix('.pdf'))
plt.savefig(out_pdf, dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

### Version 4 all six glaciers HS, AH, CH, SR, GT, CV + glacier outlines + hillshade on top ortho

### Function to create switzerland inset

In [ ]:
def add_switzerland_inset(
    ax,
    overview_extent,
    inset_position=(0.72, -0.02, 0.26, 0.26),
    buffer_km=40,
    shp_path="/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/products/figures/maps/World_Countries_(Generalized)_-573431906301700955/World_Countries_Generalized.shp",
):
    """
    Add a small inset map of Switzerland showing the location of the overview extent,
    using the locally stored World Countries (Generalized) shapefile.
    """
    from matplotlib.patches import Rectangle
    from shapely.geometry import box
    import geopandas as gpd

    world = gpd.read_file(shp_path)
    # Find ISO 2-letter field
    iso_col = next((c for c in ["ISO2", "ISO_A2", "ISO"] if c in world.columns), None)
    if iso_col is None:
        raise RuntimeError("ISO country code column not found in shapefile.")

    world = world.to_crs(epsg=2056)

    ax_inset = ax.inset_axes(inset_position)

    # Add "CH" label
    ax_inset.patch.set_facecolor("white")
    ax_inset.patch.set_alpha(0.9)
    ax_inset.text(0.5, 0.5, "CH", transform=ax_inset.transAxes,
                  ha="center", va="center", fontsize=14, fontweight="bold",
                  color="black", zorder=4)

    ax_inset.patch.set_facecolor("white")
    ax_inset.patch.set_alpha(0.9)

    # Switzerland extent and buffer
    ch_extent = (2480000, 2840000, 1070000, 1300000)
    buf = buffer_km * 1000
    bbox_geom = box(ch_extent[0] - buf, ch_extent[2] - buf,
                    ch_extent[1] + buf, ch_extent[3] + buf)
    bbox_poly = gpd.GeoSeries([bbox_geom], crs="EPSG:2056")

    neighbors = gpd.clip(world, bbox_poly.iloc[0])
    ch = neighbors[neighbors[iso_col] == "CH"]

    if not neighbors.empty:
        neighbors.plot(ax=ax_inset, facecolor="#e8e8e8", edgecolor="gray", linewidth=0.6, zorder=1)
    if not ch.empty:
        ch.plot(ax=ax_inset, facecolor="lightgray", edgecolor="black", linewidth=0.9, zorder=2)

    # Red rectangle for the provided overview_extent
    rect = Rectangle(
        (overview_extent[0], overview_extent[2]),
        overview_extent[1] - overview_extent[0],
        overview_extent[3] - overview_extent[2],
        linewidth=1.8, edgecolor="red", facecolor="none", zorder=5,
    )
    ax_inset.add_patch(rect)

    ax_inset.set_xlim(ch_extent[0] - buf, ch_extent[1] + buf)
    ax_inset.set_ylim(ch_extent[2] - buf, ch_extent[3] + buf)
    ax_inset.set_aspect("equal")
    ax_inset.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax_inset.spines.values():
        spine.set_edgecolor("black"); spine.set_linewidth(1)
    return ax_inset

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

overview_path = "/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/projects/polythermal_swiss_glaciers/products/figures/maps/field_sites_extended_overview.png"

abbr = {"Alphubel": "AH", "Chessjen": "CJ", "Hohsaas": "HS", "Sex Rouge": "SR", "Tortin": "GT", "Corvatsch": "CV"}
rect_colors = {"AH": "#1f77b4", "CJ": "#2ca02c", "HS": "#ff7f0e", "SR": "#d62728", "GT": "#9467bd", "CV": "#8c564b"}

ANNO_FONTSIZE = 12
ABBR_FONTSIZE = 12

fig = plt.figure(figsize=(13, 12), dpi=300)
gs = fig.add_gridspec(3, 3, height_ratios=[0.6, 1.0, 1.0], hspace=0.0)

extent = (np.float64(2481120.0), np.float64(2838080.0), np.float64(1071040.0), np.float64(1300160.0))

ax_a = fig.add_subplot(gs[0, 0:2])
_, _, cropped_extent = plot_cropped_map_with_grid(
    overview_path, extent, crop_top_px=900, grid_interval_km=50,
    ax=ax_a, fontsize=ANNO_FONTSIZE, xlabel=False, ylabel=False,
)

ax_inset = add_switzerland_inset(
    ax_a, overview_extent=cropped_extent,
    inset_position=(0.68, 0.055, 0.42, 0.42), buffer_km=40
)

ax_a.text(0.01, 0.96, "(a)", transform=ax_a.transAxes, ha='left', va='top',
          fontsize=ANNO_FONTSIZE, fontweight='bold', color='black',
          bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=1.0), zorder=21)

ax_legend = fig.add_subplot(gs[0, 2])
ax_legend.set_axis_off()
ax_legend.set_xlim(0, 1); ax_legend.set_ylim(0, 1)

sym_x = 0.1; text_x = 0.2; y0 = 0.78; dy = 0.12  # <-- moved up

ax_legend.text(0.2, 0.97, "Legend", transform=ax_legend.transAxes,  # <-- title moved up
               fontsize=ANNO_FONTSIZE, fontweight='bold', va='top', ha='center')

y_positions = [y0 - i * dy for i in range(7)]

# Glacier area (blue square)
ax_legend.add_patch(Rectangle((sym_x - 0.015, y_positions[0] - 0.02), 0.03, 0.03, transform=ax_legend.transAxes,
                             facecolor='#8fb5e0', edgecolor='k', linewidth=0.8, zorder=2))
ax_legend.text(text_x, y_positions[0], "Glacierized area (SGI 2022)", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, va='center', ha='left')

# Study site (diamond)
ax_legend.scatter([sym_x], [y_positions[1]], s=90, marker='D', color='red',
                  edgecolors='black', linewidths=1.0, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[1], "Study site", transform=ax_legend.transAxes,
               fontsize=ANNO_FONTSIZE, va='center', ha='left')

# Geoprecision borehole (filled red circle)
ax_legend.scatter([sym_x], [y_positions[2]], s=90, marker='o', color='red',
                  edgecolors='black', linewidths=1.5, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[2], "Geoprecision borehole", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# TinyTag borehole (white circle)
ax_legend.scatter([sym_x], [y_positions[3]], s=90, marker='o', color='white',
                  edgecolors='black', linewidths=1.5, transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[3], "TinyTag borehole", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# GPR profile (line)
ax_legend.plot([sym_x - 0.06, sym_x + 0.06], [y_positions[4], y_positions[4]], color='k', lw=2.5,
               solid_capstyle='round', transform=ax_legend.transAxes)
ax_legend.text(text_x, y_positions[4], "GPR profile", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# Glacier outline (SGI 2022)
ax_legend.plot([sym_x - 0.06, sym_x + 0.06], [y_positions[5], y_positions[5]],
               color='darkblue', lw=2.5, linestyle='-', transform=ax_legend.transAxes, zorder=2)
ax_legend.text(text_x, y_positions[5], "Glacier outline (SGI 2022)", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# Flow direction arrow (solid block: thick body + wide triangular head)
from matplotlib.patches import FancyArrow
_leg_arrow = FancyArrow(
    sym_x - 0.06, y_positions[6], 0.12, 0,
    width=0.025, head_width=0.06, head_length=0.04,
    length_includes_head=True, color='red', zorder=2,
    transform=ax_legend.transAxes
)
ax_legend.add_patch(_leg_arrow)
ax_legend.text(text_x, y_positions[6], "Flow direction", transform=ax_legend.transAxes,
               va='center', ha='left', fontsize=ANNO_FONTSIZE)

# North arrow
add_north_arrow_compass(ax_legend, xy=(0.92, 0.85), size=0.1, colors=('white', '0.35'), edge='0.35', text_color='0.35', lw=1.0)

# Middle row: b, c, d
axs_mid = [fig.add_subplot(gs[1, i]) for i in range(3)]
draw_glacier_map(axs_mid[0], out_ortho_AH, bbox_AH, gpr_lines_AH, dem_tiles_AH, boreholes_AH, "Alphubel", xlabel=False, ylabel=False, panel='b', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_AH, background='ortho_hillshade', flow_arrow_angle_deg=225,flow_arrow_pos_offset=(20, -35))
draw_glacier_map(axs_mid[1], out_ortho_CJ, bbox_CJ, gpr_lines_CJ, dem_tiles_CJ, boreholes_CJ, "Chessjen", xlabel=False, ylabel=False, panel='c', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_CJ, background='ortho_hillshade', flow_arrow_angle_deg=75)
draw_glacier_map(axs_mid[2], out_ortho_HS, bbox_HS, gpr_lines_HS, dem_tiles_HS, boreholes_HS, "Hohsaas", xlabel=False, ylabel=False, panel='d', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_HS, background='ortho_hillshade',flow_arrow_angle_deg=190)

# Bottom row: e, f, g
axs_bot = [fig.add_subplot(gs[2, i]) for i in range(3)]
draw_glacier_map(axs_bot[0], out_ortho_SR, bbox_SR, gpr_lines_SR, dem_tiles_SR, boreholes_SR, "Sex Rouge", xlabel=False, ylabel=False, panel='e', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_SR, background='ortho_hillshade',flow_arrow_pos_offset=(20, 0))
draw_glacier_map(axs_bot[1], out_ortho_GT, bbox_GT, gpr_lines_GT, dem_tiles_GT, boreholes_GT, "Tortin", xlabel=False, ylabel=False, panel='f', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_TT, background='ortho_hillshade',flow_arrow_angle_deg=120,flow_arrow_pos_offset=(30, 0))
draw_glacier_map(axs_bot[2], out_ortho_CV, bbox_CV, gpr_lines_CV, dem_tiles_CV, boreholes_CV, "Corvatsch", xlabel=False, ylabel=False, panel='g', ANNO_FONTSIZE=ANNO_FONTSIZE, ABBR_FONTSIZE=ABBR_FONTSIZE, outlines=outline_CV, background='ortho_hillshade',flow_arrow_angle_deg=40)

plt.tight_layout()

fig.text(0.55, 0.1, "Easting (LV95) [m]", ha='center', va='bottom', fontsize=12)
fig.text(0.1, 0.55, "Northing (LV95) [m]", ha='center', va='center', rotation='vertical', fontsize=12)

fig.subplots_adjust(left=0.13, bottom=0.13)

for ax, bbox in zip(axs_mid + axs_bot, [bbox_HS, bbox_AH, bbox_CJ, bbox_SR, bbox_GT, bbox_CV]):
    add_panel_outline(ax, bbox, color='black', linewidth=1.4)

out_pdf = str(Path(out_fig).with_suffix('.pdf'))
plt.savefig(out_pdf, dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

### Compress figures to max 5 mb

In [ ]:
# Define output paths
field_sites_overview_dir = output_path + "field_sites_overview_map.pdf"

# compress figures
compress_figure_inplace(field_sites_overview_dir)

## Compute geometrical properties of field sites

In [ ]:
# compute geometric properties for different field sites
props_AH = geometric_properties_from_xyzn_demtiles(xyzn_path_AH, dem_tiles_AH) # Alphubel
props_CJ = geometric_properties_from_xyzn_demtiles(xyzn_path_CJ, dem_tiles_CJ) # Chessjen
props_HS = geometric_properties_from_xyzn_demtiles(xyzn_path_HS, dem_tiles_HS) # Hohsaas
props_SR = geometric_properties_from_xyzn_demtiles(xyzn_path_SR, dem_tiles_SR) # Sex Rouge
props_GT = geometric_properties_from_xyzn_demtiles(xyzn_path_TT, dem_tiles_GT) # Tortin
props_CV = geometric_properties_from_xyzn_demtiles(xyzn_path_CV, dem_tiles_CV) # Corvatsch

In [ ]:
# props_AH
# props_CJ
# props_HS
# props_SR
props_GT
# props_CV

In [ ]:
# --- Plotting: Ice thickness maps using draw_glacier_map, gprp.imshow_grid, and gprp.plot_thickness_contours ---
import matplotlib.pyplot as plt

# Generate colormap
cmap_blue = gprp.cairomakie_cmap("Blues", n=256, reverse=False)

# Global levels if you want consistent colors across glaciers
levels_all = gprp.thickness_levels([grid_AH, grid_CJ, grid_HS], step=5.0)
norm = BoundaryNorm(levels_all, cmap_blue.N, clip=True)

# Define config handles if not already present
PANEL_GAP     = 0.0
FIG_EDGE_PAD  = 0.00
CB_POS        = [0.17, -0.1, 0.66, 0.032]
CB_LABEL_FONTSIZE = 18
CB_TICK_FONTSIZE  = 18

fig, axs = plt.subplots(1, 3, figsize=(11.5, 3.8), dpi=300, constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=FIG_EDGE_PAD, h_pad=FIG_EDGE_PAD,
                                wspace=PANEL_GAP, hspace=0.0)

for ax, ortho_path, grid, tfm, bbox, gdf_pts, dem_tiles, boreholes, outline, title in [
    (axs[0], out_ortho_AH, grid_AH, grid_transform_AH, bbox_AH, gpr_lines_AH, dem_tiles_AH, boreholes_AH, outline_AH, "Alphubel"),
    (axs[1], out_ortho_CJ, grid_CJ, grid_transform_CJ, bbox_CJ, gpr_lines_CJ, dem_tiles_CJ, boreholes_CJ, outline_CJ, "Chessjen"),
    (axs[2], out_ortho_HS, grid_HS, grid_transform_HS, bbox_HS, gpr_lines_HS, dem_tiles_HS, boreholes_HS, outline_HS, "Hohsaas"),
]:
    if ax is axs[1] or ax is axs[2]:
        ylabel = False
    else:
        ylabel = True

    # Draw glacier map background and features
    draw_glacier_map(
        ax=ax,
        ortho_path=ortho_path,
        bbox=bbox,
        gdf_pts=gdf_pts,
        dem_tiles=dem_tiles,
        boreholes=boreholes,
        title=title,
        ABBR_FONTSIZE=ABBR_FONTSIZE,
        ANNO_FONTSIZE=ANNO_FONTSIZE,
        background="hillshade",
        ylabel=ylabel,
        outlines=outline,
        show_borehole_labels=False,
    )
    # Plot ice thickness raster on top
    im, _ = gprp.imshow_grid(ax, grid, tfm, cmap=cmap_blue, alpha=0.9, norm=norm, zorder=5)
    # Plot thickness contours on top
    gprp.plot_thickness_contours(ax, grid, tfm, levels=levels_all, step=10.0, zorder_minor=6, zorder_major=7)

# Shared colorbar
cax = fig.add_axes(CB_POS)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap_blue), cax=cax, orientation='horizontal')
cb.set_label('Ice thickness [m]', fontsize=CB_LABEL_FONTSIZE)
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

out_fig = os.path.join(output_path, "ice_thickness_maps.pdf")
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Plotting: Ice thickness maps using draw_glacier_map, gprp.imshow_grid, and gprp.plot_thickness_contours ---
import matplotlib.pyplot as plt

# Generate colormap
cmap_blue = gprp.cairomakie_cmap("Blues", n=256, reverse=False)

# Global levels if you want consistent colors across glaciers
levels_all = gprp.thickness_levels([grid_AH, grid_CJ, grid_HS, grid_SR, grid_GT, grid_CV], step=5.0)
norm = BoundaryNorm(levels_all, cmap_blue.N, clip=True)

# Define config handles if not already present
PANEL_GAP     = 0.0
FIG_EDGE_PAD  = 0.00
CB_POS        = [0.17, -0.1, 0.66, 0.032]
CB_LABEL_FONTSIZE = 18
CB_TICK_FONTSIZE  = 18

fig, axs = plt.subplots(2, 3, figsize=(11, 7), dpi=300, constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.0, h_pad=FIG_EDGE_PAD, wspace=0.0, hspace=0.0)  # No gaps at all

# List of all glaciers and their plotting info
glacier_panels = [
    # Row 1
    (axs[0,0], out_ortho_AH, grid_AH, grid_transform_AH, bbox_AH, gpr_lines_AH, dem_tiles_AH, boreholes_AH, outline_AH, "Alphubel", "a"),
    (axs[0,1], out_ortho_CJ, grid_CJ, grid_transform_CJ, bbox_CJ, gpr_lines_CJ, dem_tiles_CJ, boreholes_CJ, outline_CJ, "Chessjen", "b"),
    (axs[0,2], out_ortho_HS, grid_HS, grid_transform_HS, bbox_HS, gpr_lines_HS, dem_tiles_HS, boreholes_HS, outline_HS, "Hohsaas", "c"),
    # Row 2
    (axs[1,0], out_ortho_SR, grid_SR, grid_transform_SR, bbox_SR, gpr_lines_SR, dem_tiles_SR, boreholes_SR, outline_SR, "Sex Rouge", "d"),
    (axs[1,1], out_ortho_GT, grid_GT, grid_transform_GT, bbox_GT, gpr_lines_GT, dem_tiles_GT, boreholes_GT, outline_TT, "Tortin", "e"),
    (axs[1,2], out_ortho_CV, grid_CV, grid_transform_CV, bbox_CV, gpr_lines_CV, dem_tiles_CV, boreholes_CV, outline_CV, "Corvatsch", "f"),
]

highlight_ids = {
    "Alphubel": [4],
    "Chessjen": [47],
    "Hohsaas": [40,39],
    "Sex Rouge": [],
    "Tortin": [],
    "Corvatsch": [],
}

for i, (ax, ortho_path, grid, tfm, bbox, gdf_pts, dem_tiles, boreholes, outline, title, panel) in enumerate(glacier_panels):
    # Only show y-label for first column
    show_ylabel = (i % 3 == 0)
    # Set up annotation dict for each glacier
    annotations = {}
    if title == "Alphubel":
        annotations["4"] = {"text": "L4", "color": "crimson", "fontsize": ANNO_FONTSIZE+2}
    elif title == "Chessjen":
        annotations["47"] = {"text": "L47", "color": "crimson", "fontsize": ANNO_FONTSIZE+2}
    elif title == "Hohsaas":
        annotations["39"] = {"text": "L39", "color": "crimson", "fontsize": ANNO_FONTSIZE+2}
        annotations["40"] = {"text": "L40", "color": "crimson", "fontsize": ANNO_FONTSIZE+2}
    draw_glacier_map(
        ax=ax,
        ortho_path=ortho_path,
        bbox=bbox,
        gdf_pts=gdf_pts,
        dem_tiles=dem_tiles,
        boreholes=boreholes,
        title=title,
        ABBR_FONTSIZE=ABBR_FONTSIZE,
        ANNO_FONTSIZE=ANNO_FONTSIZE,
        background="hillshade",
        outlines=outline,
        show_borehole_labels=False,
        ylabel=show_ylabel,
        highlight_ids=highlight_ids.get(title, None),
        annotations=annotations,
        panel=panel,
        show_bedrock_depth=True,
    )
    # Plot ice thickness raster on top
    im, _ = gprp.imshow_grid(ax, grid, tfm, cmap=cmap_blue, alpha=0.9, norm=norm, zorder=5)
    # Plot thickness contours on top
    gprp.plot_thickness_contours(ax, grid, tfm, levels=levels_all, step=10.0, zorder_minor=6, zorder_major=7)
    # Remove only the x-axis label from upper row (keep tick labels)
    if i < 3:
        ax.set_xlabel('')

# Shared colorbar
cax = fig.add_axes(CB_POS)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap_blue), cax=cax, orientation='horizontal')
cb.set_label('Ice thickness [m]', fontsize=CB_LABEL_FONTSIZE)
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE)

out_fig = os.path.join(output_path, "ice_thickness_maps.pdf")
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()

### Switch Hohsaas to Gabriela's data

In [ ]:
# Define paths
gpr_data_dir = '/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/gpr/'
gpr_output_dir = '/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/gpr/20250928_Hohsaas_data_drone_gpr/centerline_picks/'
ice_thickness_tiff = '/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/gpr/20250928_Hohsaas_data_drone_gpr/topography/hohsaas_ice_thickness.tif'

In [ ]:
import pandas as pd
import numpy as np

def merge_gpr_line_data(coords_file, surface_file, bed_file, line_name, output_file):
    """
    Merge GPR line coordinates with surface and bed elevation data.
    Creates format compatible with load_points_from_csv with thickness pre-calculated.
    
    Parameters
    ----------
    coords_file : str
        Path to CSV file with columns: begin, end, begin_2, end_2, xcoord, ycoord
    surface_file : str
        Path to text file with surface elevations (one per line)
    bed_file : str
        Path to text file with bed elevations (one per line)
    line_name : str
        Name of the profile/line (e.g., "centerline1", "centerline2")
    output_file : str
        Path where the merged CSV should be saved
    
    Returns
    -------
    df : pd.DataFrame
        Merged dataframe with all information
    """
    
    # Load coordinates
    coords_df = pd.read_csv(coords_file)
    
    # Load surface elevations (remove quotes if present)
    with open(surface_file, 'r') as f:
        surface_elevs = [float(line.strip().strip('"')) for line in f if line.strip()]
    
    # Load bed elevations
    with open(bed_file, 'r') as f:
        bed_elevs = [float(line.strip().strip('"')) for line in f if line.strip()]
    
    # Check that all arrays have same length
    n_coords = len(coords_df)
    n_surface = len(surface_elevs)
    n_bed = len(bed_elevs)
    
    print(f"Line {line_name}:")
    print(f"  Coordinates: {n_coords} points")
    print(f"  Surface: {n_surface} values")
    print(f"  Bed: {n_bed} values")
    
    # Use the minimum length to avoid indexing errors
    n_points = min(n_coords, n_surface, n_bed)
    
    if n_coords != n_surface or n_coords != n_bed:
        print(f"  WARNING: Lengths don't match! Using first {n_points} points.")
    
    # Create ONE row per point with thickness and bed elevation filled
    results = []
    for i in range(n_points):
        surface_elev = surface_elevs[i]
        bed_elev = bed_elevs[i]
        thickness = surface_elev - bed_elev  # Calculate ice thickness
        
        results.append({
            'Profile': line_name,
            'Channel': '',
            'Interpretation': 'Horizon interpretation',  # Match target format
            'Trace number': i + 1,
            'X': coords_df.loc[i, 'xcoord'],
            'Y': coords_df.loc[i, 'ycoord'],
            'Depth': thickness,           # Ice thickness (surface - bed)
            'Travel time': '',
            'Elevation': bed_elev,        # Bed elevation (NOT surface!)
            'Amplitude': ''
        })
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    # Save to CSV
    df.to_csv(output_file, index=False)
    print(f"  Saved {len(df)} points to: {output_file}\n")
    
    return df


def merge_multiple_lines(line_configs, output_combined):
    """
    Merge multiple GPR lines into a single file.
    
    Parameters
    ----------
    line_configs : list of dict
        Each dict should have keys: 'coords_file', 'surface_file', 'bed_file', 'line_name'
    output_combined : str
        Path where the combined CSV should be saved
    
    Returns
    -------
    df_all : pd.DataFrame
        Combined dataframe with all lines
    """
    
    all_dfs = []
    
    for config in line_configs:
        # Process each line individually
        temp_output = config.get('temp_output', None)
        if temp_output is None:
            temp_output = f"/tmp/{config['line_name']}_temp.csv"
        
        df = merge_gpr_line_data(
            coords_file=config['coords_file'],
            surface_file=config['surface_file'],
            bed_file=config['bed_file'],
            line_name=config['line_name'],
            output_file=temp_output
        )
        
        all_dfs.append(df)
    
    # Combine all lines
    df_all = pd.concat(all_dfs, ignore_index=True)
    
    # Save combined file
    df_all.to_csv(output_combined, index=False)
    print(f"Combined file saved to: {output_combined}")
    print(f"Total points: {len(df_all)} ({len(line_configs)} lines)")
    
    return df_all


# Example usage for Hohsaas glacier with 2 lines
line_configs = [
    {
        'line_name': 'centerline1',
        'coords_file': gpr_data_dir + '20250928_Hohsaas_data_drone_gpr/flights_and_profiles/centerline_1_three_bhs.csv',
        'surface_file': gpr_data_dir + '20250928_Hohsaas_data_drone_gpr/centerline/centerline_1_glacier_surface.txt',
        'bed_file': gpr_data_dir + '20250928_Hohsaas_data_drone_gpr/centerline/centerline_1_bed.txt',
        'temp_output': gpr_output_dir + '20250928_Hohsaas_centerline_1_picks.csv'
    },
    {
        'line_name': 'centerline2',
        'coords_file': gpr_data_dir + '20250928_Hohsaas_data_drone_gpr/flights_and_profiles/centerline_2_three_bhs.csv',
        'surface_file': gpr_data_dir + '20250928_Hohsaas_data_drone_gpr/centerline/centerline_2_glacier_surface.txt',
        'bed_file': gpr_data_dir + '20250928_Hohsaas_data_drone_gpr/centerline/centerline_2_bed.txt',
        'temp_output': gpr_output_dir + '20250928_Hohsaas_centerline_2_picks.csv'
    }
]

# Process all lines and create combined file
df_combined = merge_multiple_lines(
    line_configs=line_configs,
    output_combined=gpr_output_dir + '20250928_Hohsaas_drone_gpr_centerline_picks.csv'
)

# Display sample of the result
print("\nSample of combined data:")
print(df_combined.head(20))

# Load combined GPR line data
drone_gpr_lines_hohsaas = gpr.load_points_from_csv(gpr_output_dir + '20250928_Hohsaas_drone_gpr_centerline_picks.csv', epsg=2056, source_epsg=2056, drop_duplicates=True, aggregate_duplicates='mean')

In [ ]:
# --- Plotting: Ice thickness maps using draw_glacier_map, gprp.imshow_grid, and gprp.plot_thickness_contours ---
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

# Generate colormap
cmap_blue = gprp.cairomakie_cmap("Blues", n=256, reverse=False)

# Global levels if you want consistent colors across glaciers
levels_all = gprp.thickness_levels([grid_AH, grid_CJ, grid_HS, grid_SR, grid_GT, grid_CV], step=5.0)
norm = BoundaryNorm(levels_all, cmap_blue.N, clip=True)

# Define config handles if not already present
PANEL_GAP     = 0.0
FIG_EDGE_PAD  = 0.00
CB_LABEL_FONTSIZE = 18
CB_TICK_FONTSIZE  = 18

fig, axs = plt.subplots(2, 3, figsize=(11, 7), dpi=300, constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.0, h_pad=FIG_EDGE_PAD, wspace=0.0, hspace=0.0)

# List of all glaciers and their plotting info
glacier_panels = [
    # Row 1
    (axs[0,0], out_ortho_AH, grid_AH, grid_transform_AH, bbox_AH, gpr_lines_AH, None, dem_tiles_AH, boreholes_AH, outline_AH, "Alphubel", "a"),
    (axs[0,1], out_ortho_CJ, grid_CJ, grid_transform_CJ, bbox_CJ, gpr_lines_CJ, None, dem_tiles_CJ, boreholes_CJ, outline_CJ, "Chessjen", "b"),
    (axs[0,2], out_ortho_HS, grid_HS, grid_transform_HS, bbox_HS, gpr_lines_HS, drone_gpr_lines_hohsaas, dem_tiles_HS, boreholes_HS, outline_HS, "Hohsaas", "c"),
    # Row 2
    (axs[1,0], out_ortho_SR, grid_SR, grid_transform_SR, bbox_SR, gpr_lines_SR, None, dem_tiles_SR, boreholes_SR, outline_SR, "Sex Rouge", "d"),
    (axs[1,1], out_ortho_GT, grid_GT, grid_transform_GT, bbox_GT, gpr_lines_GT, None, dem_tiles_GT, boreholes_GT, outline_TT, "Tortin", "e"),
    (axs[1,2], out_ortho_CV, grid_CV, grid_transform_CV, bbox_CV, gpr_lines_CV, None, dem_tiles_CV, boreholes_CV, outline_CV, "Corvatsch", "f"),
]

highlight_ids = {
    "Alphubel": [4],
    "Chessjen": [47],
    "Hohsaas": [],
    "Sex Rouge": [],
    "Tortin": [],
    "Corvatsch": [],
}

for i, (ax, ortho_path, grid, tfm, bbox, gdf_pts, gdf_pts_2, dem_tiles, boreholes, outline, title, panel) in enumerate(glacier_panels):
    # Only show y-label for first column
    show_ylabel = (i % 3 == 0)
    
    # Set up annotation dict for each glacier
    annotations = {}
    if title == "Alphubel":
        annotations["4"] = {"text": "L4", "color": "crimson", "fontsize": ANNO_FONTSIZE+2}
    elif title == "Chessjen":
        annotations["47"] = {"text": "L47", "color": "crimson", "fontsize": ANNO_FONTSIZE+2}
    
    # Draw glacier map with optional second GPR dataset
    draw_glacier_map(
        ax=ax,
        ortho_path=ortho_path,
        bbox=bbox,
        gdf_pts=gdf_pts,
        gdf_pts_2=gdf_pts_2,
        dem_tiles=dem_tiles,
        boreholes=boreholes,
        title=title,
        ABBR_FONTSIZE=ABBR_FONTSIZE,
        ANNO_FONTSIZE=ANNO_FONTSIZE,
        background="hillshade",
        outlines=outline,
        show_borehole_labels=False,
        ylabel=show_ylabel,
        highlight_ids=highlight_ids.get(title, None),
        annotations=annotations,
        panel=panel,
        show_bedrock_depth=True,
        show_flow_arrow=False,
    )
    
    # Plot ice thickness raster on top; use external TIFF for Hohsaas
    if title == "Hohsaas":
        im, _ = gprp.imshow_grid(ax, ice_thickness_tiff, cmap=cmap_blue, alpha=0.9, norm=norm, zorder=5)
    else:
        im, _ = gprp.imshow_grid(ax, grid, tfm, cmap=cmap_blue, alpha=0.9, norm=norm, zorder=5)
    
    # Plot thickness contours on top; use external TIFF for Hohsaas
    _minor_kw = {'linewidths': 0.25, 'colors': 'k', 'alpha': 0.4}
    _major_kw = {'linewidths': 0.6, 'colors': 'k', 'alpha': 0.55}
    if title == "Hohsaas":
        gprp.plot_thickness_contours(ax, ice_thickness_tiff, levels=levels_all, step=10.0, zorder_minor=6, zorder_major=7, minor_kwargs=_minor_kw, major_kwargs=_major_kw)
    else:
        gprp.plot_thickness_contours(ax, grid, tfm, levels=levels_all, step=10.0, zorder_minor=6, zorder_major=7, minor_kwargs=_minor_kw, major_kwargs=_major_kw)
    
    # Remove only the x-axis label from upper row (keep tick labels)
    if i < 3:
        ax.set_xlabel('')

# ========== MANUALLY ADD ANNOTATIONS AFTER ALL PLOTTING ==========

# Helper function to find profile END position (last point) with optional offset
def get_profile_end(gdf_pts, profile_id, offset_x=0, offset_y=0):
    """
    Get end coordinates of a profile in data coordinates with optional offset.
    """
    sub = gdf_pts[gdf_pts['profile'] == profile_id].copy()
    if sub.empty:
        return None
    
    # Order by PCA to get actual end point
    coords = np.c_[sub.geometry.x.values, sub.geometry.y.values]
    if coords.shape[0] < 2:
        x, y = coords[0, 0], coords[0, 1]
    else:
        mean = coords.mean(0)
        _, _, Vt = np.linalg.svd(coords - mean, full_matrices=False)
        t = (coords - mean) @ Vt[0]
        idx = np.argsort(t)[-1]  # Last point along principal direction
        x, y = sub.iloc[idx].geometry.x, sub.iloc[idx].geometry.y
    
    # Apply offset
    return x + offset_x, y + offset_y

# Alphubel L4 annotation (at end of profile)
prof4_pos = get_profile_end(gpr_lines_AH, 4, offset_x=10, offset_y=10)
if prof4_pos:
    axs[0,0].text(prof4_pos[0], prof4_pos[1], "L4",
                  fontsize=ANNO_FONTSIZE, fontweight='bold', color='crimson',
                  ha='center', va='center', zorder=100,
                  bbox=dict(boxstyle='round,pad=0.25', facecolor='white', 
                           edgecolor='crimson', linewidth=1.2, alpha=0.9))

# Chessjen L47 annotation (at end of profile)
prof47_pos = get_profile_end(gpr_lines_CJ, 47, offset_y=10)
if prof47_pos:
    axs[0,1].text(prof47_pos[0], prof47_pos[1], "L47",
                  fontsize=ANNO_FONTSIZE, fontweight='bold', color='crimson',
                  ha='center', va='center', zorder=100,
                  bbox=dict(boxstyle='round,pad=0.25', facecolor='white', 
                           edgecolor='crimson', linewidth=1.2, alpha=0.9))

# Hohsaas L1 annotation (at end of profile)
profL1_pos = get_profile_end(drone_gpr_lines_hohsaas, 1, offset_x=-3, offset_y=20)
if profL1_pos:
    axs[0,2].text(profL1_pos[0], profL1_pos[1], "L1",
                  fontsize=ANNO_FONTSIZE, fontweight='bold', color='darkorange',
                  ha='center', va='center', zorder=100,
                  bbox=dict(boxstyle='round,pad=0.25', facecolor='white', 
                           edgecolor='darkorange', linewidth=1.2, alpha=0.9))
                           
# Hohsaas L2 annotation (at end of profile)
profL2_pos = get_profile_end(drone_gpr_lines_hohsaas, 2, offset_x=15, offset_y=-20)
if profL2_pos:
    axs[0,2].text(profL2_pos[0], profL2_pos[1], "L2",
                  fontsize=ANNO_FONTSIZE, fontweight='bold', color='darkorange',
                  ha='center', va='center', zorder=100,
                  bbox=dict(boxstyle='round,pad=0.25', facecolor='white', 
                           edgecolor='darkorange', linewidth=1.2, alpha=0.9))

# Create unified legend box spanning the top
legend_box_pos = [0.046, 1.05, 0.927, 0.18]  # [left, bottom, width, height]

# Add background box for entire legend
legend_box = FancyBboxPatch(
    (legend_box_pos[0], legend_box_pos[1]), legend_box_pos[2], legend_box_pos[3],
    boxstyle="square,pad=0.01",
    transform=fig.transFigure,
    facecolor='none', edgecolor='black', linewidth=1,
    zorder=100
)
fig.add_artist(legend_box)

# Colorbar in left portion of legend box
CB_POS = [0.055, 1.14, 0.5, 0.032]
cax = fig.add_axes(CB_POS)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap_blue), cax=cax, orientation='horizontal')
cb.set_label('Ice thickness [m]', fontsize=CB_LABEL_FONTSIZE-4)
cb.ax.tick_params(labelsize=CB_TICK_FONTSIZE-2)

# Add text title for legend
fig.text(0.09, 1.19, "Legend", ha='center', va='bottom', fontsize=CB_LABEL_FONTSIZE-4, fontweight='bold')

# Legend items in right portion
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
           markeredgecolor='black', markersize=7, linewidth=0,
           label='Geoprecision borehole'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white', 
           markeredgecolor='black', markersize=7, linewidth=0,
           label='TinyTag borehole'),
    Line2D([0], [0], color='black', linewidth=2.5, linestyle='-',
           label='Ground-based GPR'),
    Line2D([0], [0], color='darkorange', linewidth=2.5, linestyle='-',
           label='Drone-based GPR (centerlines)'),
    Line2D([0], [0], color='darkblue', linewidth=2.5, linestyle='-',
           label='Glacier outline (SGI 2022)'),
]

legend = fig.legend(
    handles=legend_elements, 
    loc='upper right',
    bbox_to_anchor=(0.95, 1.25),
    fontsize=CB_LABEL_FONTSIZE-5,
    frameon=False,
    title_fontproperties={'weight': 'bold', 'size': CB_LABEL_FONTSIZE-2},
    ncol=1
)

out_fig = os.path.join(output_path, "ice_thickness_maps_with_dronegpr.pdf")
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()

## Compress ice thickness maps 

In [ ]:
# Define output paths
ice_thickness_maps_dir = output_path + "ice_thickness_maps_with_dronegpr.pdf"

# compress figures
compress_figure_inplace(ice_thickness_maps_dir)

### Get GPR bedrock picks and generate profiles 

In [ ]:
# Build a profile table (use the raw TXT files)
df_all_AH = gpr.load_points_from_txt(gpr_picks_dirs[0], epsg=2056, drop_duplicates=True, aggregate_duplicates='mean', return_type='df')

# Choose a profile id present in df_all['profile'] (e.g., 12)
prof_id_AH = 4
prof_AH = gpr.extract_profile_table(df_all_AH, prof_id_AH, order_method="pca")

# Build a profile table for Chessjen (use the raw CSV files)
df_all_cj = gpr.load_points_from_csv(gpr_picks_dirs[1], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean', return_type='df')

# Choose a profile id present in df_all_cj['profile'] (e.g., 47)
prof_id_cj = 47
prof_cj = gpr.extract_profile_table(df_all_cj, prof_id_cj, order_method="pca")

# Build a profile table for Hohsaas (use the raw CSV files)
df_all_hs = gpr.load_points_from_csv(gpr_picks_dirs[2], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean', return_type='df')

# Choose a profile id present in df_all_hs['profile'] (e.g., 40)
prof_hs1 = gpr.extract_profile_table(df_all_hs, 39, order_method="pca")
prof_hs2 = gpr.extract_profile_table(df_all_hs, 40, order_method="pca")

### Compute some general statistics on ice thickness results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def compute_gpr_statistics(gpr_dataframes, glacier_names):
    """
    Compute and visualize ice thickness statistics from multiple GPR dataframes.
    
    Parameters
    ----------
    gpr_dataframes : list of GeoDataFrame
        List of GPR dataframes (e.g., [gpr_lines_AH, gpr_lines_CJ, gpr_lines_HS])
    glacier_names : list of str
        Corresponding glacier names (e.g., ['Alphubel', 'Chessjen', 'Hohsaas'])
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure with statistics plots
    stats_df : pd.DataFrame
        DataFrame containing all statistics
    """
    
    # Collect statistics for each glacier
    stats = []
    for gdf, name in zip(gpr_dataframes, glacier_names):
        thickness = gdf['thickness'].dropna()
        
        stats.append({
            'Glacier': name,
            'N_points': len(thickness),
            'N_profiles': gdf['profile'].nunique(),
            'Mean [m]': thickness.mean(),
            'Median [m]': thickness.median(),
            'Std [m]': thickness.std(),
            'Min [m]': thickness.min(),
            'Max [m]': thickness.max(),
            'Q25 [m]': thickness.quantile(0.25),
            'Q75 [m]': thickness.quantile(0.75),
        })
    
    stats_df = pd.DataFrame(stats)
    
    # Create figure with multiple panels
    fig = plt.figure(figsize=(14, 10), dpi=150)
    gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)
    
    # Panel 1: Box plots
    ax1 = fig.add_subplot(gs[0, :])
    data_to_plot = [gdf['thickness'].dropna().values for gdf in gpr_dataframes]
    bp = ax1.boxplot(data_to_plot, labels=glacier_names, patch_artist=True,
                     showmeans=True, meanline=False,
                     medianprops=dict(color='red', linewidth=2),
                     meanprops=dict(marker='D', markerfacecolor='blue', markeredgecolor='blue', markersize=6))
    
    for patch, color in zip(bp['boxes'], ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd', '#8c564b'][:len(glacier_names)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax1.set_ylabel('Ice thickness [m]', fontsize=12)
    ax1.set_title('Ice Thickness Distribution by Glacier', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.text(0.02, 0.98, '(a)', transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top')
    
    # Panel 2: Histograms
    ax2 = fig.add_subplot(gs[1, :])
    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd', '#8c564b'][:len(glacier_names)]
    
    for gdf, name, color in zip(gpr_dataframes, glacier_names, colors):
        thickness = gdf['thickness'].dropna()
        ax2.hist(thickness, bins=30, alpha=0.5, label=name, color=color, edgecolor='black', linewidth=0.5)
    
    ax2.set_xlabel('Ice thickness [m]', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title('Ice Thickness Frequency Distribution', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right', fontsize=10)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.text(0.02, 0.98, '(b)', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top')
    
    # Panel 3: Statistics table
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.axis('off')
    
    # Select key statistics for table
    table_data = stats_df[['Glacier', 'N_points', 'Mean [m]', 'Median [m]', 'Min [m]', 'Max [m]']].copy()
    table_data['Mean [m]'] = table_data['Mean [m]'].round(1)
    table_data['Median [m]'] = table_data['Median [m]'].round(1)
    table_data['Min [m]'] = table_data['Min [m]'].round(1)
    table_data['Max [m]'] = table_data['Max [m]'].round(1)
    
    table = ax3.table(cellText=table_data.values,
                     colLabels=table_data.columns,
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Style header
    for i in range(len(table_data.columns)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Alternate row colors
    for i in range(1, len(table_data) + 1):
        for j in range(len(table_data.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#D9E2F3')
    
    ax3.text(0.5, 1.05, '(c) Summary Statistics', transform=ax3.transAxes, 
             fontsize=12, fontweight='bold', ha='center')
    
    # Panel 4: Bar chart of mean thickness
    ax4 = fig.add_subplot(gs[2, 1])
    x_pos = np.arange(len(glacier_names))
    means = stats_df['Mean [m]'].values
    stds = stats_df['Std [m]'].values
    
    bars = ax4.bar(x_pos, means, yerr=stds, capsize=5, alpha=0.7, 
                   color=colors, edgecolor='black', linewidth=1)
    
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(glacier_names, rotation=45, ha='right')
    ax4.set_ylabel('Mean ice thickness [m]', fontsize=12)
    ax4.set_title('Mean Ice Thickness (± 1σ)', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3, linestyle='--', axis='y')
    ax4.text(0.02, 0.98, '(d)', transform=ax4.transAxes, fontsize=12, fontweight='bold', va='top')
    
    plt.tight_layout()
    
    return fig, stats_df

# Usage in your notebook:
glacier_names = ['Alphubel', 'Chessjen', 'Hohsaas', 'Sex Rouge', 'Tortin', 'Corvatsch']
gpr_dataframes = [gpr_lines_AH, gpr_lines_CJ, gpr_lines_HS, gpr_lines_SR, gpr_lines_GT, gpr_lines_CV]

fig_stats, stats_table = compute_gpr_statistics(gpr_dataframes, glacier_names)

# Print statistics table
print("\nIce Thickness Statistics:")
print("="*80)
print(stats_table.to_string(index=False))
print("="*80)

# Save figure
out_stats = os.path.join(output_path, "gpr_ice_thickness_statistics.pdf")
fig_stats.savefig(out_stats, dpi=300, bbox_inches='tight')
plt.show()

# Optionally save statistics as CSV
stats_csv = os.path.join(output_path, "gpr_ice_thickness_statistics.csv")
stats_table.to_csv(stats_csv, index=False)
print(f"\nStatistics saved to: {stats_csv}")